# 06 – Early Warning System for Prediction Market Efficiency


Previous analysis revealed that market categories do not explain forecast accuracy.

Instead, trajectory-based information dynamics such as:

* Max Drawdown
* Probability Range
* Reversals
* Volatility

were found to be strongly associated with market efficiency.

However, all previous analyses used the complete probability trajectory of each market.

This introduces an important limitation:

Market participants do not have access to the future.

In practice, traders and researchers must evaluate a market while it is still active.

The objective of this notebook is therefore to determine whether market efficiency can be identified early in the market lifecycle.

---

## Research Question

Can we predict whether a prediction market will ultimately be efficient using only the first portion of its probability trajectory?

---

## Hypothesis

### Null Hypothesis

Early trajectory information does not contain predictive information about eventual market efficiency.

$  H_0 $

Early market dynamics are unrelated to final forecast accuracy.

---

### Alternative Hypothesis

Early trajectory information contains predictive information about eventual market efficiency.

$ H_1 $

Markets reveal signals of future efficiency long before resolution.

---

## Methodology

For each market, only the first fraction of the probability trajectory will be used.

Three information horizons will be analyzed:

### Horizon 1

First 25% of market life

### Horizon 2

First 50% of market life

### Horizon 3

First 75% of market life

---

## Early-Trajectory Features

For each horizon, the following features will be computed:

### Probability Range

Measures the extent of belief revisions.

$ Range = \max(P_t)-\min(P_t) $

### Max Drawdown

Measures the largest correction in market beliefs.

### Realized Volatility

Measures cumulative probability movement.

### Reversals

Measures disagreement and directional changes.

### Trend

Measures the dominant direction of market belief evolution.

### Shannon Entropy

Measures uncertainty and information complexity.

---

## Target Variable

Market efficiency is defined using forecast error:

$ AbsSurprise = |Outcome - FinalProbability| $ 

Markets below the median forecast error are classified as:

$  Efficient = 1 $

Markets above the median forecast error are classified as:

$  Efficient = 0 $

---

## Modeling Framework

A Logistic Regression classifier will be trained using only early-market information.

Model performance will be evaluated using:

* Stratified K-Fold Cross Validation
* Accuracy
* F1 Score

---

## Evaluation Criteria

The objective is to determine how early market efficiency becomes predictable.

Expected interpretation:

| Horizon | Interpretation                    |
| ------- | --------------------------------- |
| 25%     | Very early signal detection       |
| 50%     | Mid-life efficiency forecasting   |
| 75%     | Late-stage efficiency forecasting |

---

## Success Criterion

A model that achieves predictive performance significantly above random classification using only early trajectory information would suggest that market efficiency can be identified before market resolution.

Such a result would indicate that information incorporation dynamics reveal the quality of a prediction market long before the final outcome is known.

---

## Expected Contribution

This notebook extends the previous findings by moving from:

**Explaining market efficiency**

to

**Forecasting market efficiency before resolution**

which is substantially closer to a real-world trading and quantitative research setting.


In [13]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

prices = pd.read_csv("../data/raw/prices_raw.csv")
df_model_v2 = pd.read_csv("../data/processed/df_model_v2.csv")

prices["timestamp"] = pd.to_datetime(prices["timestamp"])
prices = prices.sort_values(["market_id", "timestamp"])

In [14]:
from scipy.stats import entropy, skew, kurtosis, linregress

def compute_features_for_slice(df_slice):
    df_slice = df_slice.sort_values("timestamp").copy()
    df_slice["prob_return"] = df_slice["implied_probability"].diff()
    p = df_slice["implied_probability"].dropna()
    r = df_slice["prob_return"].dropna()
    if len(p) < 5:
        return pd.Series({
            "early_realized_volatility": np.nan,
            "early_probability_range": np.nan,
            "early_trend": np.nan,
            "early_max_drawdown": np.nan,
            "early_reversals": np.nan,
            "early_entropy": np.nan,
        })
    realized_volatility = np.sum(r**2)
    probability_range = p.max() - p.min()
    x = np.arange(len(p))
    trend = linregress(x, p).slope
    running_max = np.maximum.accumulate(p)
    drawdown = (running_max - p) / running_max.replace(0, np.nan)
    max_drawdown = np.nanmax(drawdown)
    signs = np.sign(r)
    reversals = np.sum(signs.diff().fillna(0) != 0)
    if len(r) >= 10:
        hist, _ = np.histogram(r, bins=10)
        probs = hist / hist.sum()
        probs = probs[probs > 0]
        shannon_entropy = entropy(probs)
    else:
        shannon_entropy = np.nan
    return pd.Series({
        "early_realized_volatility": realized_volatility,
        "early_probability_range": probability_range,
        "early_trend": trend,
        "early_max_drawdown": max_drawdown,
        "early_reversals": reversals,
        "early_entropy": shannon_entropy,
    })


def build_early_features(prices, horizon=0.25):
    rows = []
    for market_id, g in prices.groupby("market_id"):
        g = g.sort_values("timestamp").copy()
        n = len(g)
        cutoff = max(int(n * horizon), 5)
        g_early = g.iloc[:cutoff]
        features = compute_features_for_slice(g_early)
        features["market_id"] = market_id
        features["horizon"] = horizon
        rows.append(features)
    return pd.DataFrame(rows)

features_25 = build_early_features(prices, horizon=0.25)
features_50 = build_early_features(prices, horizon=0.50)
features_75 = build_early_features(prices, horizon=0.75)

In [15]:
features_25.head()

,early_realized_volatility,early_probability_range,early_trend,early_max_drawdown,early_reversals,early_entropy,market_id,horizon
0,0.543700,0.870,-0.001323,0.935484,32.0,0.094190,248594.0,0.25
1,0.000000,0.000,0.000000,0.000000,0.0,NaN,249778.0,0.25
2,0.005625,0.075,-0.015000,0.187500,1.0,NaN,250474.0,0.25
3,0.222800,0.695,0.006473,0.609929,25.0,1.032947,251025.0,0.25
4,0.087475,0.240,-0.000068,0.304054,13.0,0.725695,251027.0,0.25


In [16]:
features_25.shape, features_50.shape, features_75.shape

((43, 8), (43, 8), (43, 8))

In [17]:
target = df_model_v2[["market_id","abs_surprise"]].copy()

threshold = target["abs_surprise"].median()

target["efficient_market"] = (target["abs_surprise"] <= threshold).astype(int)

In [21]:
early_25 = features_25.merge(target, on="market_id", how="left")
early_50 = features_50.merge(target, on="market_id", how="left")
early_75 = features_75.merge(target, on="market_id", how="left")

early_25.to_csv('../data/processed/easly_25.csv', index = False )
early_50.to_csv('../data/processed/easly_50.csv', index = False )
early_75.to_csv('../data/processed/easly_75.csv', index = False )

In [19]:
early_features = [
    "early_realized_volatility",
    "early_probability_range",
    "early_trend",
    "early_max_drawdown",
    "early_reversals",
    "early_entropy"
]


def evaluate_early_model(df, horizon_name):
    X = df[early_features]
    y = df["efficient_market"]
    pipe = Pipeline(steps=[("imputer", SimpleImputer(strategy="median")),("scaler", StandardScaler()),
                ("model", LogisticRegression(max_iter=1000)) ])
    cv = StratifiedKFold(n_splits=5,shuffle=True, random_state=42)

    acc_scores = cross_val_score(pipe, X, y,cv=cv,scoring="accuracy")

    f1_scores = cross_val_score(pipe,X,y,cv=cv,scoring="f1")

    return {
        "horizon": horizon_name,
        "accuracy_mean": acc_scores.mean(),
        "accuracy_std": acc_scores.std(),
        "f1_mean": f1_scores.mean(),
        "f1_std": f1_scores.std()
    }

In [20]:
results = pd.DataFrame([evaluate_early_model(early_25, "25%"),evaluate_early_model(early_50, "50%"),
                        evaluate_early_model(early_75, "75%")])

results

,horizon,accuracy_mean,accuracy_std,f1_mean,f1_std
0,25%,0.675000,0.138221,0.701111,0.108889
1,50%,0.675000,0.128380,0.701111,0.108889
2,75%,0.672222,0.138778,0.726566,0.096442


Early trajectory dynamics contain most of the information required to identify whether a prediction market will ultimately become efficient.

Market efficiency can be detected surprisingly early.
Using only the first 25% of market life,
the model achieves approximately 67.5% accuracy,
compared with 74.7% when using the complete trajectory.